# 당뇨병 데이터 분석 — 답지용(solution)

scikit-learn 당뇨병 데이터셋을 분석하는 실습.
심화 EDA → 피처 엔지니어링 → 데이터 증강(엄격 비교) → 다중 모델·튜닝 → 모델 해석의 전체 파이프라인 구성.

**실습 방법**: `# TODO` 빈칸(`______`)을 채운 뒤 셀 실행. 막히면 답지용과 비교.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import skew, kurtosis
from IPython.display import display

# 한글 폰트 자동 선택 (Mac/Linux/Win 호환)
for cand in ["AppleGothic", "NanumGothic", "Noto Sans CJK KR", "Malgun Gothic", "NanumBarunGothic"]:
    if any(cand in f.name for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = cand
        break
plt.rcParams["axes.unicode_minus"] = False

from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.ensemble import (RandomForestRegressor, GradientBoostingRegressor,
                              HistGradientBoostingRegressor, IsolationForest)
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import (train_test_split, KFold, RepeatedKFold,
                                     cross_val_score, RandomizedSearchCV, learning_curve)
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.mixture import GaussianMixture

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

RANDOM_STATE = 42
print("준비 완료")

KeyboardInterrupt: 

## 1. 데이터 로드 및 품질 점검

결측·중복 점검과 요약 통계로 데이터 상태 파악.

In [ ]:
# 데이터 로드 및 품질 점검
# scaled=False → 정규화를 풀어 원본 스케일로 로드 (age=세, sex=1/2, bmi/bp 등 실제 단위)
data = load_diabetes(scaled=False, as_frame=True)
X = data.data.copy()
y = data.target.copy()

# sex: 1.0/2.0 두 범주 → 0/1 이진 인코딩 (모델 입력용)
# 주의: sklearn은 1/2 의 실제 성별을 공개하지 않아 '그룹 A/B' 로 표기
X["sex"] = (X["sex"] == 2.0).astype(int)

df = X.copy(); df["target"] = y
print("형태:", df.shape)
print("결측치 합계:", int(df.isna().sum().sum()), "| 중복행:", int(df.duplicated().sum()))
print("age 범위:", X["age"].min(), "~", X["age"].max(), "세 | sex 분포:", dict(X["sex"].value_counts()))
display(df.head())
display(df.describe().T.round(2))

## 2. 심화 탐색적 분석(EDA)

### 2.1 타깃 분포와 정규성

왜도·첨도로 분포의 치우침 정량화.

In [ ]:
# 타깃 분포와 정규성 점검
print(f"왜도(skew) = {skew(y):.3f}  |  첨도(kurtosis) = {kurtosis(y):.3f}")
fig = px.histogram(df, x="target", nbins=30, marginal="box",
                   title="타깃(1년 후 진행도) 분포")
fig.show()

### 2.2 성별·나이 분석 (정규화 해제 후 범주형/실수형 복원)

정규화 상태로는 해석 불가했던 sex(범주형)와 age(나이, 세)를 원본 스케일로 복원해 개별 분석. sex는 1/2 → 그룹 A/B 범주로, age는 연령대로 구간화해 타깃과의 관계 확인.

In [ ]:
# 성별(범주형) EDA — 정규화를 풀어 범주로 복원한 변수
df_eda = df.copy()
df_eda["성별"] = X["sex"].map({0: "그룹 A", 1: "그룹 B"})
print("성별 분포:", dict(df_eda["성별"].value_counts()))
display(df_eda.groupby("성별")["target"].agg(["count", "mean", "median", "std"]).round(1))
px.box(df_eda, x="성별", y="target", color="성별", points="all",
       title="성별 그룹별 타깃 분포").show()

In [ ]:
# 나이(연속형) EDA — 원본 스케일(세)
px.histogram(df, x="age", nbins=25, marginal="box", title="나이 분포 (세)").show()
px.scatter(df, x="age", y="target", trendline="ols", opacity=0.5,
           color=df_eda["성별"], title="나이 vs 타깃 (성별 구분)").show()

# 연령대 구간화 → 파생 범주형 변수
df_eda["연령대"] = pd.cut(X["age"], bins=[0, 40, 50, 60, 120],
                        labels=["~30대", "40대", "50대", "60대+"])
display(df_eda.groupby("연령대")["target"].agg(["count", "mean"]).round(1))
px.box(df_eda, x="연령대", y="target", color="연령대",
       title="연령대별 타깃 분포").show()

### 2.3 타깃 구간별 피처 분포

타깃을 사분위로 나눠 피처가 구간별로 어떻게 달라지는지 확인.

In [ ]:
# 타깃 사분위 구간별 핵심 피처 분포 (바이올린)
dfq = df.copy()
dfq["타깃구간"] = pd.qcut(dfq["target"], 4, labels=["Q1(낮음)", "Q2", "Q3", "Q4(높음)"])
for f in ["bmi", "s5", "bp", "s3"]:
    px.violin(dfq, x="타깃구간", y=f, box=True, points=False,
              title=f"{f} — 타깃 사분위별 분포").show()

### 2.4 상관 구조와 다중공선성

계층 클러스터맵으로 유사 변수 군집 확인 후, VIF로 공선성 진단.

In [ ]:
# 상관관계 계층 클러스터맵 (유사 변수 군집 확인)
cg = sns.clustermap(df.corr(), annot=True, fmt=".2f", cmap="RdBu_r",
                    center=0, figsize=(9, 9))
cg.fig.suptitle("상관관계 계층 클러스터맵", y=1.02)
plt.show()

In [ ]:
# 다중공선성 진단: VIF (10 이상이면 강한 공선성)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

Xc = add_constant(X)
vif = pd.DataFrame({
    "변수": X.columns,
    "VIF": [variance_inflation_factor(Xc.values, i + 1) for i in range(X.shape[1])]
}).sort_values("VIF", ascending=False).reset_index(drop=True)
display(vif.round(2))

혈청 지표(s1·s2 등)는 서로 강하게 연관되어 VIF가 높게 나타남 — 정규화 모델이 유리한 근거.

### 2.5 비선형 의존성(상호정보량)

선형 상관이 못 잡는 비선형 관계를 MI로 보완.

In [ ]:
# 상호정보량(MI): 선형 상관이 놓치는 비선형 의존성 포착
mi = pd.Series(mutual_info_regression(X, y, random_state=RANDOM_STATE),
               index=X.columns).sort_values(ascending=False)
px.bar(x=mi.values, y=mi.index, orientation="h",
       title="상호정보량(MI) — 타깃과의 비선형 의존성",
       labels={"x": "MI", "y": "변수"}).show()

### 2.6 차원 축소 및 이상치 탐지

PCA로 구조를 2D로 압축하고 IsolationForest로 이상치 식별.

In [ ]:
# PCA 2차원 투영 (타깃으로 색칠)
Xs = StandardScaler().fit_transform(X)
pca = PCA(n_components=2).fit(Xs)
pcs = pca.transform(Xs)
pdf = pd.DataFrame(pcs, columns=["PC1", "PC2"]); pdf["target"] = y.values
px.scatter(pdf, x="PC1", y="PC2", color="target", color_continuous_scale="Viridis",
           title=f"PCA 2D (누적 설명분산 {pca.explained_variance_ratio_.sum():.1%})").show()

In [ ]:
# 이상치 탐지: IsolationForest
iso = IsolationForest(contamination=0.05, random_state=RANDOM_STATE).fit(Xs)
flag = iso.predict(Xs)
print("이상치 탐지:", int((flag == -1).sum()), f"/ {len(X)} 건")
pdf["판정"] = np.where(flag == -1, "이상치", "정상")
px.scatter(pdf, x="PC1", y="PC2", color="판정",
           title="IsolationForest 이상치 (PCA 평면)").show()

## 3. 피처 엔지니어링

EDA에서 영향력이 큰 bmi·s5를 중심으로 상호작용·비선형 파생변수 생성.

In [ ]:
# 피처 엔지니어링: 시각화에서 확인한 핵심 변수(bmi, s5) 기반 파생변수 생성
def add_features(d_in):
    d = d_in.copy()
    d["bmi_s5"] = d["bmi"] * d["s5"]      # 핵심 두 변수의 상호작용
    d["bmi_bp"] = d["bmi"] * d["bp"]      # 비만 × 혈압
    d["s5_bp"]  = d["s5"]  * d["bp"]
    d["tc_hdl_gap"] = d["s1"] - d["s3"]   # 총콜레스테롤 - HDL
    d["bmi_sq"] = d["bmi"] ** 2           # 비선형(2차) 항
    return d

Xfe = add_features(X)
new_cols = ["bmi_s5", "bmi_bp", "s5_bp", "tc_hdl_gap", "bmi_sq"]
print("원본 피처:", X.shape[1], "→ 파생 후:", Xfe.shape[1])

# 파생변수의 타깃 상관 확인
corr_new = pd.concat([Xfe[new_cols], y], axis=1).corr()["target"].drop("target").sort_values(ascending=False)
display(corr_new.round(3).to_frame("타깃 상관"))

## 4. 모델링: 여러 모델 비교

선형 ~ 부스팅까지 13종 회귀 모델(XGBoost·LightGBM 포함)을 동일한 5-fold 교차검증으로 학습하고, R²·RMSE·MAE 다중 지표로 비교.

In [ ]:
# 13종 회귀 모델 라인업 (선형 ~ 부스팅, XGBoost·LightGBM 포함)
def sc(m):
    return make_pipeline(StandardScaler(), m)

models = {
    "LinearRegression": sc(LinearRegression()),
    "Ridge": sc(Ridge(alpha=1.0)),
    "Lasso": sc(Lasso(alpha=0.1, max_iter=10000)),
    "ElasticNet": sc(ElasticNet(alpha=0.1, l1_ratio=0.5, max_iter=10000)),
    "SVR": sc(SVR(C=100, gamma="scale")),
    "KNN": sc(KNeighborsRegressor(n_neighbors=15)),
    "DecisionTree": DecisionTreeRegressor(max_depth=4, random_state=RANDOM_STATE),
    "RandomForest": RandomForestRegressor(n_estimators=400, random_state=RANDOM_STATE),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=400, random_state=RANDOM_STATE),
    "GradientBoosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "HistGBM": HistGradientBoostingRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=3,
                            subsample=0.9, random_state=RANDOM_STATE, verbosity=0),
    "LightGBM": LGBMRegressor(n_estimators=400, learning_rate=0.05,
                              random_state=RANDOM_STATE, verbose=-1),
}

# 성능 평가 지표: R²(클수록 좋음), RMSE·MAE(작을수록 좋음)
scoring = {"R2": "r2", "RMSE": "neg_root_mean_squared_error", "MAE": "neg_mean_absolute_error"}
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, m in models.items():
    r = cross_validate(m, Xfe, y, cv=cv, scoring=scoring)
    rows.append({"모델": name, "R²": r["test_R2"].mean(), "R² 표준편차": r["test_R2"].std(),
                 "RMSE": -r["test_RMSE"].mean(), "MAE": -r["test_MAE"].mean()})
cmp = pd.DataFrame(rows).sort_values("R²", ascending=False).reset_index(drop=True)
cmp.insert(0, "순위", cmp.index + 1)
display(cmp.round(4))

In [ ]:
# 모델별 성능 비교 시각화
px.bar(cmp, x="모델", y="R²", error_y="R² 표준편차", color="R²",
       color_continuous_scale="Tealgrn", title="모델별 5-fold 교차검증 R² (높을수록 좋음)").show()
px.bar(cmp.sort_values("RMSE"), x="모델", y="RMSE", color="RMSE",
       color_continuous_scale="OrRd_r", title="모델별 RMSE (낮을수록 좋음)").show()

## 5. 데이터 증강(뻥튀기)과 엄격한 비교

442건은 적은 편이라 학습 데이터 증강을 시도. **핵심 원칙**: 증강은 학습 폴드에만 적용하고 테스트 폴드는 항상 원본 유지 → 데이터 누수 차단. 증강이 실제로 성능을 올리는지 동일 교차검증으로 정직하게 비교.

In [ ]:
# 데이터 증강 함수 정의 (학습 데이터 뻥튀기)
# 가우시안 노이즈: 기존 표본에 잡음을 더해 유사 표본 생성(jittering)
def aug_gaussian(Xtr, ytr, n_new, noise=0.5, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.randint(0, len(Xtr), n_new)
    Xn = Xtr.values[idx] + rng.normal(0, noise, (n_new, Xtr.shape[1])) * Xtr.values.std(0)
    yn = ytr.values[idx] + rng.normal(0, noise, n_new) * ytr.values.std()
    return pd.DataFrame(Xn, columns=Xtr.columns), pd.Series(yn)

# GMM 합성: 가우시안 혼합으로 분포를 학습한 뒤 새 표본을 샘플링
def aug_gmm(Xtr, ytr, n_new, n_comp=8, seed=0):
    Z = np.column_stack([Xtr.values, ytr.values])
    gm = GaussianMixture(n_components=n_comp, covariance_type="full", random_state=seed).fit(Z)
    samp, _ = gm.sample(n_new)
    return pd.DataFrame(samp[:, :-1], columns=Xtr.columns), pd.Series(samp[:, -1])

print("증강 함수 정의 완료")

In [ ]:
# 엄격한 비교: 증강은 '학습 폴드'에만 적용, 테스트 폴드는 항상 원본 유지 → 데이터 누수 방지
def eval_aug(Xd, aug_fn=None, mult=1.0, model_key="HistGBM", seed=RANDOM_STATE):
    cv = KFold(5, shuffle=True, random_state=seed)
    sc = []
    for tr, te in cv.split(Xd):
        Xtr, Xte = Xd.iloc[tr], Xd.iloc[te]
        ytr, yte = y.iloc[tr], y.iloc[te]
        if aug_fn is not None:
            Xa, ya = aug_fn(Xtr, ytr, int(len(Xtr) * mult))
            Xtr = pd.concat([Xtr, Xa], ignore_index=True)
            ytr = pd.concat([ytr, ya], ignore_index=True)
        m = HistGradientBoostingRegressor(random_state=RANDOM_STATE)
        m.fit(Xtr, ytr)
        sc.append(r2_score(yte, m.predict(Xte)))   # 평가는 항상 원본 테스트셋
    return np.array(sc)

res = {
    "원본(증강 없음)":     eval_aug(Xfe),
    "가우시안 노이즈 +100%": eval_aug(Xfe, aug_gaussian, 1.0),
    "GMM 합성 +100%":      eval_aug(Xfe, aug_gmm, 1.0),
    "GMM 합성 +300%":      eval_aug(Xfe, aug_gmm, 3.0),
}
aug_df = pd.DataFrame([{"증강 방식": k, "R² 평균": v.mean(), "R² 표준편차": v.std()}
                      for k, v in res.items()])
display(aug_df.round(4))
px.bar(aug_df, x="증강 방식", y="R² 평균", error_y="R² 표준편차",
       title="데이터 증강 방식별 성능 (테스트셋은 항상 원본)").show()

> 해석 주의: 표 형식 회귀에서 합성 증강은 분포를 모방할 뿐 새로운 정보를 만들지 못해, 성능이 크게 오르지 않거나 오히려 소폭 하락하기도 함. 증강은 만능이 아니며 검증으로 확인하는 자세가 중요.

## 6. 최종 모델 선정 및 튜닝

비교 결과 교차검증 R² 최고 모델을 최종 선정하고, 트리·부스팅 계열이면 RandomizedSearchCV 로 튜닝.

In [ ]:
# 최종 모델 선정: 교차검증 R² 최고 모델 → 트리·부스팅 계열이면 RandomizedSearchCV 튜닝
PARAM_DISTS = {
    "RandomForest": {"n_estimators": [200, 400, 600], "max_depth": [None, 4, 6, 8],
                     "max_features": ["sqrt", 0.5, 1.0], "min_samples_leaf": [1, 2, 5]},
    "ExtraTrees": {"n_estimators": [200, 400, 600], "max_depth": [None, 4, 6, 8],
                   "max_features": ["sqrt", 0.5, 1.0], "min_samples_leaf": [1, 2, 5]},
    "GradientBoosting": {"n_estimators": [100, 200, 300], "learning_rate": [0.02, 0.05, 0.1],
                         "max_depth": [2, 3, 4], "subsample": [0.8, 1.0]},
    "HistGBM": {"learning_rate": [0.02, 0.05, 0.1, 0.2], "max_depth": [None, 2, 3, 4],
                "max_leaf_nodes": [15, 31, 63], "l2_regularization": [0.0, 0.1, 1.0],
                "min_samples_leaf": [10, 20, 30]},
    "XGBoost": {"n_estimators": [200, 400, 600], "learning_rate": [0.02, 0.05, 0.1],
                "max_depth": [2, 3, 4], "subsample": [0.8, 1.0],
                "colsample_bytree": [0.8, 1.0], "reg_lambda": [1, 5, 10]},
    "LightGBM": {"n_estimators": [200, 400, 600], "learning_rate": [0.02, 0.05, 0.1],
                 "num_leaves": [15, 31, 63], "max_depth": [-1, 3, 5],
                 "subsample": [0.8, 1.0], "reg_lambda": [0, 1, 5]},
}
best_name = cmp.iloc[0]["모델"]
base = models[best_name]
if best_name in PARAM_DISTS:
    search = RandomizedSearchCV(base, PARAM_DISTS[best_name], n_iter=20, cv=5,
                                scoring="r2", random_state=RANDOM_STATE, n_jobs=-1).fit(Xfe, y)
    best = search.best_estimator_
    print(f"최종 선정: {best_name} (튜닝됨) | 최적 파라미터: {search.best_params_}")
else:
    best = base
    print(f"최종 선정: {best_name} | 선형 계열이라 정규화 기본값 사용")

## 7. 모델 해석

### 7.1 순열 중요도

In [ ]:
# 최종 모델 홀드아웃 평가 + 순열 중요도
Xtr, Xte, ytr, yte = train_test_split(Xfe, y, test_size=0.2, random_state=RANDOM_STATE)
best.fit(Xtr, ytr)
pred = best.predict(Xte)
print(f"[{best_name}] 홀드아웃 R²={r2_score(yte, pred):.4f} | "
      f"RMSE={mean_squared_error(yte, pred) ** 0.5:.2f} | MAE={mean_absolute_error(yte, pred):.2f}")
pi = permutation_importance(best, Xte, yte, n_repeats=20, random_state=RANDOM_STATE)
pis = pd.Series(pi.importances_mean, index=Xfe.columns).sort_values()
px.bar(x=pis.values, y=pis.index, orientation="h",
       title=f"순열 중요도 — {best_name}", labels={"x": "중요도", "y": "변수"}).show()

### 7.2 잔차 분석

In [ ]:
# 잔차 분석 (예측값 대 잔차 — 패턴이 없어야 이상적)
resid = yte.values - pred
fig = px.scatter(x=pred, y=resid, opacity=0.6,
                 title="잔차 분석", labels={"x": "예측값", "y": "잔차"})
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.show()

### 7.3 부분의존도(PDP)

In [ ]:
# 부분의존도(PDP): 핵심 변수가 예측에 미치는 한계 효과
top3 = pis.sort_values(ascending=False).index[:3].tolist()
fig, ax = plt.subplots(figsize=(13, 4))
PartialDependenceDisplay.from_estimator(best, Xtr, top3, ax=ax)
plt.suptitle("부분의존도(PDP) — 상위 3개 변수", y=1.05)
plt.tight_layout(); plt.show()
print("PDP 대상 변수:", top3)

### 7.4 학습곡선

In [ ]:
# 학습곡선: 표본 수가 늘면 성능이 오르는지 확인 → 증강/데이터 확보 가치 판단
sizes, tr_sc, te_sc = learning_curve(
    best, Xfe, y, cv=5, scoring="r2",
    train_sizes=np.linspace(0.1, 1.0, 8), random_state=RANDOM_STATE)
fig = go.Figure()
fig.add_scatter(x=sizes, y=tr_sc.mean(1), name="학습 R²", mode="lines+markers")
fig.add_scatter(x=sizes, y=te_sc.mean(1), name="검증 R²", mode="lines+markers")
fig.update_layout(title="학습곡선", xaxis_title="학습 표본 수", yaxis_title="R²")
fig.show()

## 8. 결론 및 시사점

- 정규화를 풀어 age(나이)·sex(성별)를 해석 가능한 범주/실수로 복원 → 성별·연령대별 타깃 차이를 직접 확인
- bmi와 s5가 선형·비선형·중요도 분석 전반에서 일관되게 핵심 인자로 확인됨
- 혈청 지표 간 강한 공선성 존재 → 정규화 선형 모델 또는 트리 계열이 안정적
- 파생변수(상호작용·비선형 항)는 모델에 따라 소폭의 성능 향상 기여
- 데이터 증강은 누수 없는 비교에서 뚜렷한 개선을 주지 못함 → 표 형식 회귀에서 합성 증강의 한계 확인
- 학습곡선의 검증 성능이 평탄 → 표본 수보다 피처 정보량이 성능의 병목
- 실무 결론: 무리한 증강보다 양질의 피처 확보와 적절한 정규화·튜닝이 우선